# Uninformed Search Techniques
---
Now, we are going to explore a uninformed search technique named Breadth-First Search (BFS), applied on a graph. The implementation is based on the pseudocode provided in the lecture slides from Canva.

In [1]:
import os

from pathlib import Path
from typing import List, Dict
from abc import ABC, abstractmethod

In [2]:
# Mount my Google Drive to have acces to my files. This code snippet is only 
# for Google Colab, it won't work on your local machine.
discriminant = os.environ.get('COLAB_JUPYTER_IP', None)
dataPath = '/My Drive/UTS/data'
mount = 'G:'

if discriminant is not None:
    from google.colab import drive
    drive.mount('/content/drive')
    del mount
    mount = '/content/drive'

dataPath = Path(mount + dataPath).resolve()

In [ ]:
class Node:
    def __init__(self, node_id: str, data: Dict = {}):
        # The constructor function takes up to two values: a, unique identifier for each node, 
        # that should be a labeling integer, and a dictionary of properties that could be 
        # either empty  
        self.id = node_id
        self.data = data

    def __hash__(self) -> int:
        # Implementing __hash__ allows Node objects to be used in hash-based collections 
        # like sets and dictionaries, ensuring uniqueness is determined by the node_id.
        return hash(self.id)

    def __eq__(self, other: object) -> bool:
        # This methods check if the instance of a node is equal to another instance of a Node
        # object by checkir their ids
        if not isinstance(other, Node):
            return False
        return self.id == other.id

    def __repr__(self) -> str:
        # Shows a string representation of the node to be called using the builtin method 
        # repr(Node)
        return f"Node({self.id})"


In [ ]:
# We implement the BaseGraph interface to ensure the future Graph objects follow the rules we 
# define for them 
class BaseGraph(ABC):
    @abstractmethod
    def add_node(self, node: Node) -> None:
        """
        Adds a node to the graph.
        
        Args:
            node (Node): The node to add.
        """
        pass

    @abstractmethod
    def add_edge(self, node_1: Node, node_2: Node, weight: int = 1) -> None:
        pass

    @abstractmethod
    def get_neighbors(sef, node: Node) -> List[Node]:
        pass

In [ ]:
class MatrixGraph(BaseGraph):
    def __init__(self, verbose: bool = True):
        self.matrix: List[List[int]] = []
        self.nodes: List[Node] = []
        self.nodes_index: Dict[str, int] = {}
        self.verbose = verbose

    def messenger(self, message: str) -> None:
        if self.verbose:
            print(message)
        pass

    def add_node(self, node: Node) -> None:
        """
        Adds a node to the matrix graph and updates the adjacency matrix.
        
        Args:
            node (Node): The node to add.
        """
        if node.id in self.nodes_index:
            self.messenger(f'[WARN] - Node({node.id}) already exists!')
        
        new_node_index = len(self.nodes)

        self.nodes.append(node)
        self.nodes_index.setdefault(node.id, new_node_index)
        
        for row in self.matrix:
            row.append(0)
            
        new_row = [0] * len(self.nodes) 
        self.matrix.append(new_row)
    
    def add_edge(self, node1: Node, node2: Node, weight: int = 1, directed: bool = False):
        idx, jdx = None, None

        try:
            if idx is None and jdx is None:
                idx = self.nodes_index[node1.id]
                jdx = self.nodes_index[node2.id]
        except KeyError as err:
            raise ValueError(f"Both nodes must be added to the graph first! Missing: {err}")
        
        self.matrix[idx][jdx] = weight
        
        if not directed:
            self.matrix[jdx][idx] = weight

    def get_neighbors(self, node: Node) -> list[Node]:
        # What is the index of the node we are asking about?
        if node.id not in self.node_to_index:
            return []
            
        node_idx = self.node_to_index[node.id]
        neighbors = []
        
        # Look at this node's specific row in the matrix
        row = self.matrix[node_idx]
        
        # Iterate through the columns of this row.
        # If the value is not 0, it means there is an edge connection!
        for col_idx, weight in enumerate(row):
            if weight != 0:
                # We found a neighbor! Use col_idx to get the actual Node object
                neighbors.append(self.nodes[col_idx])
                
        return neighbors
    # --- Bonus method for visualization ---
    def print_matrix(self):
        print("    " + " ".join([n.id for n in self.nodes]))
        for i, row in enumerate(self.matrix):
            print(f"{self.nodes[i].id} | {row}")
